# 1회차 · GPU 기반 오픈소스 LLM 실행 및 성능 비교

실습 목표: RunPod A100 → Qwen3-8B(BF16) → Qwen3-8B(NF4 4-bit) → Gemma 3 4B(BF16) → 결과 비교·CSV 저장
**참고:** Gemma 3 4B는 Hugging Face 모델 페이지에서 Google 라이선스 동의 후 접근할 수 있습니다.

- Qwen: https://huggingface.co/Qwen/Qwen3-8B
- Gemma: https://huggingface.co/google/gemma-3-4b-it

## STEP 1. 환경 확인

In [1]:
# RunPod 서버·GPU 정보 확인: Jupyter Notebook에서 실행
import sys
import subprocess

print("Python 실행 파일:", sys.executable)
subprocess.run(["nvidia-smi"], check=True)

Python 실행 파일: /usr/local/bin/python
Sun Nov 22 09:18:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.112.07             Driver Version: 595.112.07     CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:0C:00.0 Off |                    0 |
| N/A   35C    P0             48W /  300W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------

CompletedProcess(args=['nvidia-smi'], returncode=0)

## STEP 2. 실습 패키지 설치

In [2]:
%pip install -U "transformers>=4.53,<5" accelerate bitsandbytes pandas sentencepiece pillow huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 8.9 MB/s  0:00:016m0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 754.0 kB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 9.1 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 12.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 11.1 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 4.2 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.2/809.2 kB 1.6 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [accelerate]1 [accelerate]s]ub]
Note: you may need to restart the kernel to use updated packages.


## STEP 3. CUDA·캐시·평가 조건 설정
이 단계에서 이후 비교에 사용하는 **공통 질문**을 지정합니다.

In [3]:
import os
from pathlib import Path
import gc
import time
import torch
import pandas as pd

# RunPod 측 모델 캐시(서버 내 저장 위치)
os.environ.setdefault("HF_HOME", "/workspace/hf-cache")
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU가 없습니다. RunPod 원격 커널을 선택하세요."
print("GPU 이름:", torch.cuda.get_device_name(0))
print("GPU 메모리(GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

QUESTION = "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."
MAX_NEW_TOKENS = 160
results = []

print("공통 질문:", QUESTION)
# Notebook과 터미널의 Hugging Face 인증 경로가 다를 때를 대비한 공통 함수
def runpod_hf_token():
    """Return the current HF token without printing it or saving it in notebook source."""
    from pathlib import Path
    import os
    from huggingface_hub import get_token

    # The user authenticated in the RunPod shell (seen at this CLI token location).
    paths = [
        Path('/workspace/.cache/huggingface/token'),
        Path('/workspace/hf-cache/token'),
        Path.home() / '.cache' / 'huggingface' / 'token',
    ]
    for path in paths:
        if path.is_file():
            token = path.read_text(encoding='utf-8').strip()
            if token:
                return token
    token = os.environ.get('HF_TOKEN') or get_token()
    if not token:
        raise RuntimeError(
            'Hugging Face 토큰을 찾지 못했습니다. RunPod 터미널에서 hf auth login 후 다시 실행하세요.'
        )
    return token


PyTorch: 2.8.0+cu129
CUDA 사용 가능: True
GPU 이름: NVIDIA A100 80GB PCIe
GPU 메모리(GB): 79.25
공통 질문: 생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘.


## STEP 4. 공통 성능 측정 함수
두 모델에 **같은 질문과 최대 생성 길이**를 사용합니다. 초당 토큰 수는 생성 전체 소요시간을 포함한 간이 측정치입니다. 토크나이저가 다르므로 서로 다른 모델의 토큰/초를 해석할 때 주의합니다.

In [4]:
def measure_text_generation(model, tokenizer_or_processor, *, model_name, precision, gemma=False):
    """텍스트 생성 후 소요 시간, 토큰 수, GPU 최대 메모리 사용량 반환."""
    # 모델별 채팅 템플릿 적용: Gemma 3 4B는 AutoProcessor 사용
    if gemma:
        messages = [{"role": "user", "content": [{"type": "text", "text": QUESTION}]}]
        inputs = tokenizer_or_processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)
        decoder = tokenizer_or_processor.tokenizer
    else:
        messages = [{"role": "user", "content": QUESTION}]
        prompt = tokenizer_or_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False  # Qwen3의 Thinking 모드를 끄고 출력 비교
        )
        inputs = tokenizer_or_processor(prompt, return_tensors="pt").to(model.device)
        decoder = tokenizer_or_processor

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=decoder.eos_token_id
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    prompt_length = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0, prompt_length:]
    answer = decoder.decode(generated_ids, skip_special_tokens=True)
    token_count = generated_ids.numel()
    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    current_gb = torch.cuda.memory_allocated() / 1024**3

    result = {
        "model": model_name,
        "precision": precision,
        "new_tokens": int(token_count),
        "elapsed_sec": round(elapsed, 3),
        "tokens_per_sec": round(token_count / elapsed, 2) if elapsed else None,
        "peak_vram_gb": round(peak_gb, 2),
        "allocated_vram_gb": round(current_gb, 2),
        "answer": answer.strip()
    }
    print("\n[모델]", model_name, "|", precision)
    print("[응답]", answer)
    print("[성능]", {k:v for k,v in result.items() if k != "answer"})
    return result


def clear_gpu_objects():
    """이전 모델 변수 삭제 후 호출: 메모리 회수 요청."""
    gc.collect()
    torch.cuda.empty_cache()
    print("정리 후 VRAM 할당(GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

## STEP 5. Qwen3-8B 모델 BF16으로 로딩

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

qwen_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(qwen_id)
model = AutoModelForCausalLM.from_pretrained(
    qwen_id,
    dtype=torch.bfloat16,
    device_map="auto"
).eval()

print("로드 완료:", qwen_id)
print("모델 할당 VRAM(GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

로드 완료: Qwen/Qwen3-8B
모델 할당 VRAM(GB): 15.26


In [7]:
# Qwen3-8B / BF16: 동일 질문 추론
results.append(measure_text_generation(
    model, tokenizer, model_name="Qwen3-8B", precision="BF16"
))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[모델] Qwen3-8B | BF16
[응답] 생성형 AI는 입력 데이터를 기반으로 새로운 내용을 창조하는 능력을 가진 인공지능입니다. 기존 인공지능은 주로 데이터를 분석하거나 특정 작업을 수행하는 데 초점을 맞추고 있습니다. 생성형 AI는 창의성과 유연성을 통해 새로운 아이디어나 콘텐츠를 만들어내는 것이 특징입니다.
[성능] {'model': 'Qwen3-8B', 'precision': 'BF16', 'new_tokens': 90, 'elapsed_sec': 4.168, 'tokens_per_sec': 21.59, 'peak_vram_gb': 15.28, 'allocated_vram_gb': 15.27}


## STEP 6. BF16 모델 제거 및 VRAM 정리
다음 양자화 모델을 올리기 전 GPU 자원을 정리합니다.

In [8]:
del model, tokenizer
clear_gpu_objects()

정리 후 VRAM 할당(GB): 0.01


## STEP 7. Qwen3-8B 모델 NF4 4-bit 양자화 로딩
NF4(4비트 가중치 저장)와 BF16(연산 시 정밀도)는 별개입니다.

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(qwen_id)
model = AutoModelForCausalLM.from_pretrained(
    qwen_id,
    quantization_config=quant_config,
    device_map="auto"
).eval()

print("로드 완료:", qwen_id, "| 4-bit NF4")
print("모델 할당 VRAM(GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

No prebuilt binary for CUDA 12.9, loading CUDA 12.8 instead. Set BNB_CUDA_VERSION to override.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

로드 완료: Qwen/Qwen3-8B | 4-bit NF4
모델 할당 VRAM(GB): 5.67


In [10]:
# Qwen3-8B / NF4 4-bit: BF16과 동일한 질문 추론
results.append(measure_text_generation(
    model, tokenizer, model_name="Qwen3-8B", precision="NF4 4-bit"
))


[모델] Qwen3-8B | NF4 4-bit
[응답] 생성형 AI는 새로운 내용을 생성하는 능력을 가진 인공지능으로, 텍스트, 이미지, 음성 등 다양한 형태의 창의적 콘텐츠를 만들어냅니다. 기존 인공지능은 주로 입력 데이터를 기반으로 예측하거나 분류하는 기능을 수행하며, 창의적인 생성은 포함하지 않습니다. 생성형 AI는 인간의 의도나 아이디어를 바탕으로 독창적인 결과물을 만들어내는 반면, 기존 AI는 기존 데이터를 기반으로 유사한 결과를 예측하거나 분석하는 데 초점을 맞춥니다.
[성능] {'model': 'Qwen3-8B', 'precision': 'NF4 4-bit', 'new_tokens': 145, 'elapsed_sec': 21.051, 'tokens_per_sec': 6.89, 'peak_vram_gb': 5.78, 'allocated_vram_gb': 5.67}


## STEP 8. NF4 모델 제거 및 VRAM 정리

In [11]:
del model, tokenizer, quant_config
clear_gpu_objects()

정리 후 VRAM 할당(GB): 0.01


## STEP 9. Gemma 3 4B IT 모델 BF16 로딩
최초 1회 Hugging Face 라이선스 동의와 인증이 필요합니다.  
1. 브라우저에서 https://huggingface.co/google/gemma-3-4b-it 접속 → 로그인 후 사용 조건 동의  
2. 터미널에서 `hf auth login` 실행 → 개인 토큰 입력(필요 시). Notebook은 이미 로그인한 토큰 파일을 찾아 모델 로딩에 전달합니다.  
3. 아래 셀 실행. 토큰을 수업 제출물이나 Notebook 코드에 적지 않습니다.

Gemma 3 4B는 멀티모달 아키텍처여서 공식 권장 클래스인 `Gemma3ForConditionalGeneration` 및 `AutoProcessor`를 사용합니다. 이번 실습은 텍스트만 입력합니다.

In [13]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

gemma_id = "google/gemma-3-4b-it"
# 2026-11-21 실습 복구: 터미널에서 승인받은 HF 토큰을 Notebook에서 명시적으로 전달.
# hf_token 값은 출력하거나 GitHub에 저장하지 않습니다.
hf_token = runpod_hf_token()
processor = AutoProcessor.from_pretrained(gemma_id, token=hf_token)
model = Gemma3ForConditionalGeneration.from_pretrained(
    gemma_id,
    token=hf_token,
    dtype=torch.bfloat16,
    device_map="auto"
).eval()

del hf_token
print("로드 완료:", gemma_id)
print("모델 할당 VRAM(GB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

... (6 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

로드 완료: google/gemma-3-4b-it
모델 할당 VRAM(GB): 8.02


In [14]:
# Gemma 3 4B / BF16: 위와 동일한 질문 추론
results.append(measure_text_generation(
    model, processor, model_name="Gemma-3-4B-IT", precision="BF16", gemma=True
))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



[모델] Gemma-3-4B-IT | BF16
[응답] 생성형 AI는 기존 인공지능과 달리 데이터를 학습하여 새로운 콘텐츠를 창작해내는 반면, 기존 인공지능은 주어진 데이터를 분석하고 패턴을 인식하여 예측이나 분류와 같은 특정 작업을 수행합니다. 즉, 생성형 AI는 '만들기'에 집중하는 반면, 기존 인공지능은 '알기'에 집중한다고 볼 수 있습니다. 이러한 차이 때문에 생성형 AI는 텍스트, 이미지, 오디오 등 다양한 형태의 콘텐츠 제작에 활용될 수 있는 반면, 기존 인공지능은 주로 데이터 분석, 자동화, 의사 결정 등 특정 분야에서 활용됩니다.
[성능] {'model': 'Gemma-3-4B-IT', 'precision': 'BF16', 'new_tokens': 137, 'elapsed_sec': 7.324, 'tokens_per_sec': 18.71, 'peak_vram_gb': 8.04, 'allocated_vram_gb': 8.02}


## STEP 10. Gemma 모델 제거 및 GPU 메모리 정리

In [15]:
del model, processor
clear_gpu_objects()

정리 후 VRAM 할당(GB): 0.01


## STEP 11. 세 가지 실험 결과 비교 및 CSV 저장
- 같은 질문의 추론 결과를 표로 정리합니다.
- `peak_vram_gb`는 **모델 로딩 이후 추론 중** PyTorch가 관찰한 최대 GPU 할당량입니다. CUDA 드라이버/다른 프로세스 점유 메모리와 다릅니다.
- **모델 선정 및 정량 평가의 심화는 2회차**에 실시합니다.

In [16]:
from IPython.display import display

comparison = pd.DataFrame(results)
show_cols = ["model", "precision", "new_tokens", "elapsed_sec", "tokens_per_sec", "peak_vram_gb", "allocated_vram_gb"]
display(comparison[show_cols])

out_dir = Path("/workspace/results")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "01_LLM_GPU_Comparison.csv"
comparison.to_csv(out_file, index=False, encoding="utf-8-sig")
print("저장 완료:", out_file)

,model,precision,new_tokens,elapsed_sec,tokens_per_sec,peak_vram_gb,allocated_vram_gb
0,Qwen3-8B,BF16,90,4.168,21.59,15.28,15.27
1,Qwen3-8B,NF4 4-bit,145,21.051,6.89,5.78,5.67
2,Gemma-3-4B-IT,BF16,137,7.324,18.71,8.04,8.02


저장 완료: /workspace/results/01_LLM_GPU_Comparison.csv


## STEP 12. 제출 전 확인
- [ ] VS Code 원격 연결 / GPU / CUDA 확인
- [ ] Qwen3-8B BF16 답변 출력
- [ ] Qwen3-8B NF4 4-bit 답변 출력
- [ ] Gemma 3 4B BF16 답변 출력
- [ ] 세 실험의 VRAM·Tokens/sec 비교표 생성
- [ ] `/workspace/results/01_LLM_GPU_Comparison.csv` 확인

**실습 질문:** 같은 Qwen 모델에서 BF16과 NF4 4-bit의 GPU 메모리 차이가 왜 발생하는가? Gemma와 Qwen의 토큰/초 수치를 곧바로 모델 성능 우열로 해석할 수 없는 이유는 무엇인가?